In [2]:
%run "./log_monitoring"

StatementMeta(, 6c485a35-fd98-4dd8-a319-816ddc271756, 5, Finished, Available, Finished, True)

In [3]:
# from pyspark.sql.types import *
# from pyspark.sql.functions import *
# from notebookutils import mssparkutils
# from datetime import datetime
# import re
# import uuid

StatementMeta(, 6c485a35-fd98-4dd8-a319-816ddc271756, 6, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import col, lit, when, isnan, monotonically_increasing_id,\
    input_file_name, regexp_extract, to_timestamp, current_timestamp,\
    col, lit, isnan, collect_list, concat_ws, upper
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.types import BooleanType
import re
from notebookutils import mssparkutils
from datetime import datetime
import uuid

StatementMeta(, 6c485a35-fd98-4dd8-a319-816ddc271756, 7, Finished, Available, Finished, False)

In [5]:
source_location='BOAFALanding'
archive_location='BOAFAArchive'
system='boafa'
table_name='transaction'
file_Name='transaction'
fileFormat='csv'
delimiter=','
rawSchema="""[
    StructField("DUE_DATE", StringType(), TRUE),
    StructField("TRANDATE", StringType(), TRUE),
    StructField("TRANSACTION_ID", StringType(), TRUE),
    StructField("TRANSACTION_TYPE", StringType(), TRUE),
    StructField("SALES_REP_ID", StringType(), TRUE),
    StructField("ENTITY_ID", StringType(), TRUE),
    StructField("CREATE_DATE", StringType(), TRUE),
    StructField("LastModifiedDate", StringType(), TRUE)
]"""

lakehouse='BOAFALakehouse'
target_schema='raw'
monitoring_schema='config'
monitoring_table='pipeline_monitoring'

StatementMeta(, 6c485a35-fd98-4dd8-a319-816ddc271756, 8, Finished, Available, Finished, False)

In [6]:
filepath=f"Files/{source_location}/{file_Name}_*.{fileFormat}"
filecheck=f"Files/{source_location}/"
archivepath=f"Files/{archive_location}/"

StatementMeta(, 6c485a35-fd98-4dd8-a319-816ddc271756, 9, Finished, Available, Finished, False)

In [7]:
def fn_file_raw_load():

 try:
    source_raw_count=0
    target_raw_count=0
    remarks = "No file to load"
    stringSchema = f'StructType({rawSchema})'
    # customSchema = eval(stringSchema)

    stringSchema = stringSchema.replace("TRUE", "True").replace("FALSE", "False")
    customSchema = eval(stringSchema)

    rawtable = f"{lakehouse}.{target_schema}.{system}_{table_name}"

    all_files = mssparkutils.fs.ls(filecheck)
    # print all files

    # Filter matching files
    csv_files = [f.path for f in all_files if file_Name + "_" in f.path]

    if csv_files:
        if fileFormat=='csv':
            df=spark.read.format(fileFormat)\
                .option("delimiter", delimiter)\
                .schema(customSchema)\
                .option("header", True).load(filepath)
            transformed_df=df.withColumn("raw_number", monotonically_increasing_id() + lit(1))\
                             .withColumn("sys_file_timestamp",to_timestamp(regexp_extract(input_file_name(), r"(\d{8}_\d{6})", 1),"yyyyMMdd_HHmmss"))\
                             .withColumn("sys_insert_timestamp", current_timestamp())\
                             .withColumn("source",regexp_extract(regexp_extract(input_file_name(), r'.*(/[^?]+)\?.*', 1),r'.*/([^/]+)',1))
            transformed_df.write.format("delta").mode("append").saveAsTable(rawtable)

            source_raw_count=df.count()
            target_raw_count=transformed_df.count()
            remarks=f"File loaded to {rawtable} table"

        elif fileFormat=='parquet':
            df = spark.read.format(fileFormat)\
                .schema(customSchema)\
                .option("header",True).load(filepath)
            transformed_df=df.withColumn("raw_number", monotonically_increasing_id() + lit(1))\
                             .withColumn("sys_file_timestamp",to_timestamp(regexp_extract(input_file_name(), r"(\d{8}_\d{6})", 1),"yyyyMMdd_HHmmss"))\
                             .withColumn("sys_insert_timestamp", current_timestamp())\
                             .withColumn("source",regexp_extract(regexp_extract(input_file_name(), r'.*(/[^?]+)\?.*', 1),r'.*/([^/]+)',1))
            transformed_df.write.format("delta").mode("append").saveAsTable(rawtable)

            source_raw_count=df.count()
            target_raw_count=transformed_df.count()
            remarks=f"File loaded to {rawtable} table"

        else:
            remarks = "Invalid File Format"
            print("Invalid File Format")
            return source_raw_count, target_raw_count, remarks

                # Archive files
        for file in csv_files:
            mssparkutils.fs.mv(file, archivepath, overwrite=True)
            print("File Archived:", file)

    return source_raw_count, target_raw_count, remarks
 except Exception:
    raise


StatementMeta(, 6c485a35-fd98-4dd8-a319-816ddc271756, 10, Finished, Available, Finished, False)

In [8]:
try:
    monitoring_table_fullname = f"{lakehouse}.{monitoring_schema}.{monitoring_table}"
    start_time = datetime.now()

    pipeline_name="file_to_raw"
    run_id= str(uuid.uuid4())
    layer_from = filecheck
    layer_to=target_schema

    log = start_pipeline_log(
        system=system,
        run_id=run_id,
        start_time=start_time,
        pipeline_name=pipeline_name,
        layer_from=layer_from,
        layer_to=layer_to,
        table_name=table_name
    )
    source_raw_count, target_raw_count, remarks = fn_file_raw_load()

    end_pipeline_log(monitoring_table_fullname, log, "Success", source_raw_count, target_raw_count, 0, remarks, '')

except Exception as e:
    error_message= str(e)
    source_raw_count=0
    target_raw_count=0

    end_pipeline_log(monitoring_table_fullname, log, "fail", source_raw_count, target_raw_count, 0, '', error_message)


StatementMeta(, 6c485a35-fd98-4dd8-a319-816ddc271756, 11, Finished, Available, Finished, False)